# Знакомство с исходными таблицами `buildings` и `flats`

Таблиц две: `buildings` - дома, `flats` - квартиры, у квартиры есть ссылка на дом. Смотрю, что в них лежит, как они связаны и что выйдет, если склеить их в один датасет.

Доступы к базе берутся из файла `.env` в папке `part1_airflow`, в репозиторий он не попадает.

In [1]:
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

/var/folders/4l/70xpw14x7kd1jvxmsc32pw1m0000gn/T/ipykernel_34465/38523503.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Подключаемся к личной базе

Исходные таблицы лежат в той же личной базе, куда потом пишут DAG, так что доступы одни и те же - `DB_DESTINATION_*` из `.env`.

In [2]:
# load_dotenv() ищет файл .env вверх по папкам, поэтому находит part1_airflow/.env
load_dotenv()

host = os.environ['DB_DESTINATION_HOST']
port = os.environ['DB_DESTINATION_PORT']
user = os.environ['DB_DESTINATION_USER']
password = os.environ['DB_DESTINATION_PASSWORD']
db_name = os.environ['DB_DESTINATION_NAME']

# quote_plus нужен, потому что в пароле бывают символы вроде @ и &, они ломают строку подключения
engine = create_engine(f'postgresql://{user}:{quote_plus(password)}@{host}:{port}/{db_name}')

## Таблица `buildings`

In [3]:
buildings = pd.read_sql('select * from buildings', engine)
buildings.head()

,id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,6220,1965,6,55.717113,37.781120,2.64,84,12,True
1,18012,2001,2,55.794849,37.608013,3.00,97,10,True
2,17821,2000,4,55.740040,37.761742,2.70,80,10,True
3,18579,2002,4,55.672016,37.570877,2.64,771,17,True
4,9293,1971,1,55.808807,37.707306,2.60,208,9,True


In [4]:
print('Строк и колонок:', buildings.shape)
buildings.dtypes

Строк и колонок: (24620, 9)


id                     int64
build_year             int64
building_type_int      int64
latitude             float64
longitude            float64
ceiling_height       float64
flats_count            int64
floors_total           int64
has_elevator            bool
dtype: object

In [5]:
buildings.describe()

,id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total
count,24620.00000,24620.000000,24620.000000,24620.000000,24620.000000,24620.000000,24620.000000,24620.000000
mean,12310.50000,1981.377295,3.153656,55.738892,37.593816,2.753557,184.882575,12.441024
std,7107.32615,22.657856,1.636435,0.101094,0.145408,0.277912,158.427509,6.271325
min,1.00000,1901.000000,0.000000,55.211460,36.864372,2.000000,1.000000,1.000000
25%,6155.75000,1965.000000,1.000000,55.668725,37.499031,2.640000,80.000000,8.000000
50%,12310.50000,1978.000000,4.000000,55.742020,37.588400,2.640000,139.000000,12.000000
75%,18465.25000,2002.000000,4.000000,55.811775,37.698833,2.900000,242.000000,16.000000
max,24620.00000,2023.000000,6.000000,56.011032,37.946411,27.000000,4455.000000,99.000000


In [6]:
# сразу смотрю, есть ли пропуски в домах
buildings.isnull().sum()

id                   0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
dtype: int64

## Таблица `flats`

`price` - это то, что потом будет предсказывать модель, а `building_id` связывает квартиру с домом.

In [7]:
flats = pd.read_sql('select * from flats', engine)
flats.head()

,id,floor,is_apartment,kitchen_area,living_area,rooms,studio,total_area,price,building_id
0,0,9,False,9.9,19.900000,1,False,35.099998,9500000,6220
1,1,7,False,0.0,16.600000,1,False,43.000000,13500000,18012
2,2,9,False,9.0,32.000000,2,False,56.000000,13500000,17821
3,3,1,False,10.1,43.099998,3,False,76.000000,20000000,18579
4,4,3,False,3.0,14.000000,1,False,24.000000,5200000,9293


In [8]:
print('Строк и колонок:', flats.shape)
flats.dtypes

Строк и колонок: (141362, 10)


id                int64
floor             int64
is_apartment       bool
kitchen_area    float64
living_area     float64
rooms             int64
studio             bool
total_area      float64
price             int64
building_id       int64
dtype: object

In [9]:
flats.describe()

,id,floor,kitchen_area,living_area,rooms,total_area,price,building_id
count,141362.000000,141362.000000,141362.000000,141362.000000,141362.000000,141362.000000,1.413620e+05,141362.000000
mean,70680.500000,7.467346,9.001579,31.056948,2.129476,62.374644,1.944162e+07,14053.665235
std,40807.838714,5.717144,5.264076,23.968640,0.994340,40.295864,6.626954e+07,6988.831066
min,0.000000,1.000000,0.000000,0.000000,1.000000,11.000000,1.100000e+01,1.000000
25%,35340.250000,3.000000,6.100000,19.000000,1.000000,39.299999,8.900000e+06,8535.250000
50%,70680.500000,6.000000,8.800000,29.400000,2.000000,53.000000,1.185000e+07,14332.000000
75%,106020.750000,10.000000,10.200000,41.400002,3.000000,72.000000,1.695000e+07,20475.000000
max,141361.000000,56.000000,203.000000,700.000000,20.000000,960.299988,9.873738e+09,24620.000000


In [10]:
flats.isnull().sum()

id              0
floor           0
is_apartment    0
kitchen_area    0
living_area     0
rooms           0
studio          0
total_area      0
price           0
building_id     0
dtype: int64

## Как связаны таблицы

Связь один-ко-многим: одному дому (`buildings.id`) соответствует много квартир (`flats.building_id`). Проверяю два момента: сколько квартир приходится на дом и не потеряю ли я квартиры при склейке.

In [11]:
flats_per_building = flats.groupby('building_id').size()
print('Домов, в которых есть квартиры:', flats_per_building.shape[0])
flats_per_building.describe()

Домов, в которых есть квартиры: 24620


count    24620.000000
mean         5.741755
std          8.127253
min          1.000000
25%          1.000000
50%          3.000000
75%          7.000000
max        551.000000
dtype: float64

In [12]:
# квартиры, у которых нет дома с таким id: при inner join они бы потерялись
no_building = ~flats['building_id'].isin(buildings['id'])
print('Квартир без дома:', int(no_building.sum()))

# дома без квартир в датасет не попадут, и это нормально: предсказываем цену квартиры
no_flats = ~buildings['id'].isin(flats['building_id'])
print('Домов без квартир:', int(no_flats.sum()))

Квартир без дома: 0
Домов без квартир: 0


## Запрос, который пойдёт в DAG

Собираю квартиры вместе с характеристиками их домов одним запросом. Беру `left join`, а не `inner join`, чтобы квартиры без дома всё равно попали в датасет: у них характеристики дома будут пустыми, а пропуски заполняются на этапе очистки.

`f.id` переименовываю в `flat_id`, потому что в таблице-результате `flats_dataset` уже будет свой `id`. Этот же запрос дальше уходит в шаг `extract` DAG `prepare_flats_dataset`.

In [13]:
sql = '''
select
    f.id as flat_id,
    f.building_id,
    f.floor,
    f.kitchen_area,
    f.living_area,
    f.rooms,
    f.is_apartment,
    f.studio,
    f.total_area,
    f.price,
    b.build_year,
    b.building_type_int,
    b.latitude,
    b.longitude,
    b.ceiling_height,
    b.flats_count,
    b.floors_total,
    b.has_elevator
from flats as f
left join buildings as b on f.building_id = b.id
'''

dataset = pd.read_sql(sql, engine)
dataset.head()

,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.9,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.0,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.0,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.1,43.099998,3,False,False,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.0,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True


In [14]:
print('Размер результата:', dataset.shape)
print('Строк в flats было:', flats.shape[0])

# left join не должен ни размножить, ни потерять квартиры
assert dataset.shape[0] == flats.shape[0], 'после join изменилось число строк'
assert dataset['flat_id'].is_unique, 'flat_id повторяется'
print('flat_id уникален, значит его можно сделать ключом таблицы flats_dataset')

Размер результата: (141362, 18)
Строк в flats было: 141362
flat_id уникален, значит его можно сделать ключом таблицы flats_dataset


In [15]:
pd.DataFrame({'тип': dataset.dtypes, 'пропусков': dataset.isnull().sum()})

,тип,пропусков
flat_id,int64,0
building_id,int64,0
floor,int64,0
kitchen_area,float64,0
living_area,float64,0
rooms,int64,0
is_apartment,bool,0
studio,bool,0
total_area,float64,0
price,int64,0


In [16]:
dataset.sample(5, random_state=42)

,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
123548,123548,4386,6,7.400000,22.700001,2,False,False,37.599998,10950000,1962,4,55.676220,37.497215,2.64,72,9,True
123717,123717,22275,1,17.000000,15.000000,1,False,False,45.000000,10300000,2013,0,55.514637,37.374542,2.70,33,3,False
5641,5641,18752,3,34.200001,70.000000,3,False,False,121.099998,47000000,2003,2,55.672245,37.543758,3.00,145,13,True
110804,110804,24195,26,16.799999,29.000000,2,False,False,65.900002,20600000,2018,2,55.770805,37.564213,3.00,1630,44,True
70273,70273,24338,22,9.000000,18.000000,1,False,False,41.200001,12900000,2018,4,55.725483,37.743351,2.70,276,26,True


## Что я выяснил

После join получилось 18 колонок и 141 362 строки - ровно столько же, сколько квартир в `flats`. Значит, левое соединение ничего не потеряло и не размножило. `flat_id` уникален, поэтому в `flats_dataset` вешаю на него unique-ограничение: при повторном запуске DAG строки обновятся по этому ключу, а не задвоятся.

Пропусков не нашлось ни в одной колонке, поэтому и типы пришли нормальные: целые остались int64, флаги bool. Шаг `transform` в DAG всё равно оставляю: данные могут обновиться, и тогда целые колонки поедут во float, а булевы в object.

In [17]:
engine.dispose()